In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread("scenery.jpg", 0)
h, w = img.shape

a, b = -30, 30
u_noise = a + (b - a) * np.random.rand(h, w)
u_builtin = np.random.uniform(a, b, (h, w))

u1 = np.random.rand(h, w)
u2 = np.random.rand(h, w)
g_noise = 25 * np.sqrt(-2 * np.log(u1 + 1e-7)) * np.cos(2 * np.pi * u2)
g_builtin = np.random.normal(0, 25, (h, w))

k_val, a_val = 3, 0.2
prod = np.ones((h, w))
for _ in range(k_val):
    prod *= np.random.rand(h, w) + 1e-7
erlang_noise = -(1 / a_val) * np.log(prod)
erlang_noise -= np.mean(erlang_noise)
erlang_builtin = np.random.gamma(k_val, 1/a_val, (h, w)) - (k_val / a_val)

exp_noise = -(1 / 0.1) * np.log(1 - np.random.rand(h, w) + 1e-7)
exp_noise -= np.mean(exp_noise)
exp_builtin = np.random.exponential(1/0.1, (h, w)) - (1/0.1)

b_ray = 500
ray_noise = np.sqrt(-b_ray * np.log(1 - np.random.rand(h, w) + 1e-7))
ray_noise -= np.mean(ray_noise)
ray_builtin = np.random.rayleigh(np.sqrt(b_ray/2), (h, w)) - np.mean(ray_noise)

noises = [
    ("Uniform", u_noise, u_builtin),
    ("Gaussian", g_noise, g_builtin),
    ("Erlang", erlang_noise, erlang_builtin),
    ("Exponential", exp_noise, exp_builtin),
    ("Rayleigh", ray_noise, ray_builtin)
]

plt.figure(figsize=(15, 12))
for i, (name, n_calc, n_builtin) in enumerate(noises):
    corrupted = np.clip(img + n_calc, 0, 255).astype(np.uint8)
    
    plt.subplot(5, 3, 3*i + 1)
    plt.hist(n_calc.ravel(), bins=50, color='blue', alpha=0.7)
    plt.title(f"{name} Noise Histogram (Derived)")
    
    plt.subplot(5, 3, 3*i + 2)
    plt.hist(n_builtin.ravel(), bins=50, color='orange', alpha=0.7)
    plt.title(f"{name} Noise Histogram (Inbuilt)")
    
    plt.subplot(5, 3, 3*i + 3)
    plt.imshow(corrupted, cmap='gray')
    plt.title(f"Image with {name} Noise")
    plt.axis("off")

plt.tight_layout()
plt.show()
